# 多智能体共享记忆（Multi-Agent Shared Memory）

> *为协作智能体提供的共享记忆层，因为团队协作需要共同基础。*

想象厨房里的一组厨师。每位厨师都有自己的砧板（私人工作区），但大家共用台面上的一本食谱。任何厨师都可以读这本食谱，也可以在上面加注。没有这本共享的食谱，每位厨师就只能各自为战。

多智能体共享记忆对 AI 智能体来说就是同样的东西。

单个智能体处理任务时，记忆很简单。一个存储、一个读取者、一个写入者。但现代智能体系统由多个专职智能体协作组成：研究员智能体收集信息，程序员智能体写代码，审查员智能体检查质量。这些智能体需要共享信息。

研究员的发现必须传达给程序员。程序员的产出必须传达给审查员。没有共享记忆，智能体只能通过对话轮次传递一切。这样做既慢又有损（信息会在传递过程中丢失）。

多智能体共享记忆为智能体协作提供数据层。这个概念建立在经典 AI 的**黑板系统**（blackboard system）之上（Nii, 1986）。黑板系统是一个共享工作区，专职智能体把部分结果张贴上去，其他智能体读取并完善这些结果。现代实现用共享的键值存储或字典充当黑板。

**命名空间**（namespace，即 `research/`、`code/`、`review/` 这类逻辑分区）按领域组织记忆。**访问控制**规则决定哪些智能体可以读写每个分区。**冲突解决**处理两个智能体同时更新同一条目的情况。

在本 notebook 中，你将从零构建一个完整的共享记忆系统。你会创建三个智能体（研究员、程序员、审查员），通过共享记忆协作完成一个编程任务。你将看到命名空间、权限和冲突解决如何协同工作。

**读完本 notebook，你将理解：**
- 如何设计带命名空间组织的共享记忆池。
- 访问控制如何防止智能体互相覆盖工作成果。
- 冲突解决策略如何处理并发更新。
- 黑板模式如何协调多智能体协作。

## 核心概念

- **共享记忆存储（shared memory store）**：一个集中式数据存储，多个智能体都从它读取、向它写入。本 notebook 用 Python 字典充当存储。生产环境中你可能会用 Redis 或数据库。存储必须支持并发访问（多个智能体同时读写）而不损坏数据。

- **命名空间（namespace）**：共享记忆中的逻辑分区。例如 `research/`、`code/`、`review/` 就是三个命名空间。命名空间按领域组织信息，也让每个智能体的写入留在自己的区域内，从而减少冲突。

- **访问控制（access control）**：决定哪些智能体可以读写每个命名空间的规则。例如，研究员智能体可能对 `research/` 有写权限，对 `code/` 只有读权限。这能防止意外覆盖。

- **私人草稿本（private scratchpad）**：智能体的内部笔记，其他智能体看不见。可以把它想成个人笔记本，与共享白板相对。草稿本存放草稿和推理过程，只有完成的结果才进入共享记忆。

- **黑板系统（blackboard system）**：一种经典 AI 架构，智能体把部分解张贴到共享的「黑板」上。其他智能体读取、完善或扩展这些解。黑板既是通信通道，也是集体工作区。

- **冲突解决（conflict resolution）**：处理同一条目并发更新的策略。**后写胜出（last-write-wins）**保留最近的写入、丢弃较早的写入。**版本检查（version checking）**（也叫乐观锁）通过追踪版本号检测冲突，拒绝过期的写入。

- **线程安全（thread safety）**：多个线程（并行执行路径）同时访问共享数据时，保护数据不被破坏。我们用**锁**（lock，一种让同一时刻只有一个线程访问资源的机制）来保证并发访问安全。

## Architecture

下图展示私人智能体记忆如何通过访问控制层连接到共享记忆。

<p align="center">
  <img src="../../images/diagrams/22_multi_agent_shared_memory.svg" alt="架构图" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart TD
    subgraph "Agent A (Researcher)"
        PA[Private<br/>Scratchpad A]
    end

    subgraph "Agent B (Coder)"
        PB[Private<br/>Scratchpad B]
    end

    subgraph "Agent C (Reviewer)"
        PC[Private<br/>Scratchpad C]
    end

    PA --> ACL[Access Control<br/>Layer]
    PB --> ACL
    PC --> ACL

    subgraph "Shared Memory Pool"
        NS1["research/<br/>findings & sources"]
        NS2["code/<br/>artifacts & specs"]
        NS3["review/<br/>feedback & issues"]
    end

    ACL <-->|read/write| NS1
    ACL <-->|read/write| NS2
    ACL <-->|read/write| NS3

    NS1 --> CR[Conflict<br/>Resolution]
    NS2 --> CR
    NS3 --> CR
    CR --> BB[Blackboard<br/>partial solutions]

    style ACL fill:#845ef7,color:#fff
    style CR fill:#ff6b6b,color:#fff
    style BB fill:#51cf66,color:#fff
```

</details>

**数据流：**每个智能体保留一个私人草稿本用于内部推理，其他智能体看不到它。当智能体共享信息或读取他人贡献时，请求会经过访问控制层。这一层检查权限，并把读写路由到正确的命名空间。共享记忆池按领域拆分成命名空间（research、code、review）。当两个智能体写入同一个键时，冲突解决模块会选出一个胜者。黑板提供所有共享条目的统一视图。

## 环境准备

安装依赖并配置 API 访问。你需要在 `.env` 文件中设置 `LLM_API_KEY` 环境变量。

In [ ]:
%pip install -q python-dotenv deepagents langchain-openai

导入 OpenAI SDK 和标准库工具。API 密钥从 `.env` 文件加载。

In [ ]:
import os
import time
import threading
from dataclasses import dataclass, field
from enum import Enum

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 实现

我们将构建四个组件：

1. **`Permission`** 和 **`MemoryEntry`**：用于访问控制和存储条目的数据类型。
2. **`AccessControl`**：管理每个智能体对每个命名空间的读写权限。
3. **`SharedMemoryPool`**：中央存储，提供线程安全的读写和冲突解决。
4. **`Agent`**：由 LLM 驱动的智能体，带私人草稿本和共享记忆访问。

### 数据类型

`Permission` 定义三种访问级别：只读、只写和读写。`MemoryEntry` 保存共享记忆中的一条条目，记录写入者是谁，以及它的版本号。

In [ ]:
class Permission(Enum):
    """智能体在某个命名空间中的访问级别。"""
    READ = "read"
    WRITE = "write"
    READ_WRITE = "read_write"


@dataclass
class MemoryEntry:
    """共享记忆中的一条条目。"""
    namespace: str       # 例如 "research"、"code"
    key: str             # 命名空间内的标识符
    value: str           # 存储的内容
    author: str          # 写入这条目的智能体 ID
    version: int = 1     # 每次更新时递增
    timestamp: float = field(default_factory=time.time)

    def __repr__(self) -> str:
        preview = self.value[:60] + "..." if len(self.value) > 60 else self.value
        return f"MemoryEntry({self.namespace}/{self.key} v{self.version} by {self.author})"

### 访问控制层

`AccessControl` 类把每个智能体映射到它在各命名空间上被允许的操作。每次读写都要先经过这一层，才能触及共享记忆。

In [ ]:
class AccessControl:
    """管理每个智能体在每个命名空间上的权限。"""

    def __init__(self):
        self._permissions: dict[str, dict[str, Permission]] = {}

    def grant(self, agent_id: str, namespace: str, permission: Permission) -> None:
        """授予智能体对某个命名空间的特定权限。"""
        self._permissions.setdefault(agent_id, {})[namespace] = permission

    def can_read(self, agent_id: str, namespace: str) -> bool:
        """检查智能体能否从某个命名空间读取。"""
        perm = self._permissions.get(agent_id, {}).get(namespace)
        return perm in (Permission.READ, Permission.READ_WRITE)

    def can_write(self, agent_id: str, namespace: str) -> bool:
        """检查智能体能否向某个命名空间写入。"""
        perm = self._permissions.get(agent_id, {}).get(namespace)
        return perm in (Permission.WRITE, Permission.READ_WRITE)

    def describe(self) -> str:
        """返回所有权限的可读摘要。"""
        lines = []
        for agent_id, ns_perms in sorted(self._permissions.items()):
            for ns, perm in sorted(ns_perms.items()):
                lines.append(f"  {agent_id}: {ns} -> {perm.value}")
        return "\n".join(lines)

### 共享记忆池

这是中央「黑板」。它存储所有共享条目、执行权限检查，并用线程锁处理并发写入。

记忆池支持两种冲突解决策略：

- **`last_write_wins`**：最近的写入覆盖之前的任何值。速度快，但可能丢数据。
- **`version_check`**：每次写入必须提供当前版本号。如果别人抢先更新了这个条目，写入就会失败。这也叫乐观锁。

In [ ]:
class SharedMemoryPool:
    """带命名空间和冲突解决的线程安全共享记忆。"""

    def __init__(self, acl: AccessControl, conflict_strategy: str = "last_write_wins"):
        self.acl = acl
        self._store: dict[str, MemoryEntry] = {}  # 完整键 -> 条目
        self._lock = threading.Lock()
        self.conflict_strategy = conflict_strategy
        self._history: list[dict] = []  # 审计日志


`write` 方法先检查权限，再应用冲突解决策略。使用 `version_check` 时，如果调用方提供的预期版本与存储的版本不一致，写入会被拒绝。使用 `last_write_wins` 时，最新的值总是覆盖旧值。

In [ ]:
def write(
    self,
    agent_id: str,
    namespace: str,
    key: str,
    value: str,
    expected_version: int | None = None,
) -> MemoryEntry:
    """向共享记忆写入一条条目。

    先检查权限，再应用冲突策略。
    使用 version_check 策略时传入 expected_version 以检测冲突。"""
    if not self.acl.can_write(agent_id, namespace):
        raise PermissionError(
            f"Agent '{agent_id}' cannot write to namespace '{namespace}'"
        )

    full_key = f"{namespace}/{key}"

    with self._lock:
        existing = self._store.get(full_key)

        # 版本检查：版本过期时拒绝写入
        if (
            self.conflict_strategy == "version_check"
            and existing
            and expected_version is not None
            and existing.version != expected_version
        ):
            raise ValueError(
                f"Version conflict on '{full_key}': "
                f"expected v{expected_version}, found v{existing.version}"
            )

        new_version = (existing.version + 1) if existing else 1

        entry = MemoryEntry(
            namespace=namespace,
            key=key,
            value=value,
            author=agent_id,
            version=new_version,
        )
        self._store[full_key] = entry

        self._history.append({
            "action": "write",
            "agent": agent_id,
            "key": full_key,
            "version": new_version,
            "timestamp": entry.timestamp,
        })

    return entry

SharedMemoryPool.write = write

现在为 `SharedMemoryPool` 添加读取方法。
一个按命名空间和键获取特定条目。
另一个返回智能体有权限查看的所有条目。

In [ ]:
def _read(
    self, agent_id: str, namespace: str, key: str | None = None
) -> list[MemoryEntry]:
    """从命名空间读取条目。

    如果提供 key，返回该特定条目。
    如果 key 为 None，返回命名空间中的所有条目。"""
    if not self.acl.can_read(agent_id, namespace):
        raise PermissionError(
            f"Agent '{agent_id}' cannot read from namespace '{namespace}'"
        )

    with self._lock:
        if key:
            full_key = f"{namespace}/{key}"
            entry = self._store.get(full_key)
            return [entry] if entry else []
        else:
            return [
                e for e in self._store.values()
                if e.namespace == namespace
            ]

def _read_all_accessible(
    self, agent_id: str, namespaces: list[str] | None = None
) -> list[MemoryEntry]:
    """读取智能体有权限查看的所有条目。"""
    results = []
    with self._lock:
        for entry in self._store.values():
            if namespaces and entry.namespace not in namespaces:
                continue
            if self.acl.can_read(agent_id, entry.namespace):
                results.append(entry)
    return results

SharedMemoryPool.read = _read

SharedMemoryPool.read_all_accessible = _read_all_accessible


最后添加用于审计和检查的工具方法。
`get_history` 返回完整的写入日志。
`summary` 把当前记忆内容格式化后输出。

In [ ]:
def _get_history(self) -> list[dict]:
    """返回所有写操作的审计日志。"""
    return list(self._history)

def _summary(self) -> str:
    """返回共享记忆中全部内容的格式化摘要。"""
    if not self._store:
        return "(empty)"
    lines = []
    for full_key in sorted(self._store):
        entry = self._store[full_key]
        lines.append(
            f"[{full_key}] (v{entry.version}, by {entry.author}):\n"
            f"  {entry.value[:200]}"
        )
    return "\n\n".join(lines)

SharedMemoryPool.get_history = _get_history

SharedMemoryPool.summary = _summary


### 带私人草稿本的智能体

每个智能体在 LLM 调用外包了两层记忆：

1. **私人草稿本（private scratchpad）**：只有本智能体可见的笔记列表，适合内部推理和草稿。
2. **共享记忆访问**：经过访问控制层的读写操作。

智能体执行任务时，会从共享记忆收集上下文，加上自己的私人笔记，再把所有内容发给 LLM。

In [ ]:
class Agent:
    """由 LLM 驱动的智能体，带私人草稿本和共享记忆访问。"""

    def __init__(
        self,
        agent_id: str,
        role: str,
        system_prompt: str,
        shared_memory: SharedMemoryPool,
        model: str = MODEL,
    ):
        self.agent_id = agent_id
        self.role = role
        self.system_prompt = system_prompt
        self.shared_memory = shared_memory
        self.model = model
        self.scratchpad: list[str] = []  # 私有，不共享

    def think(self, thought: str) -> None:
        """向草稿本添加私人笔记。其他智能体看不到。"""
        self.scratchpad.append(thought)

    def read_shared(self, namespace: str, key: str | None = None) -> list[MemoryEntry]:
        """从共享记忆读取（带权限检查）。"""
        return self.shared_memory.read(self.agent_id, namespace, key)

    def write_shared(self, namespace: str, key: str, value: str) -> MemoryEntry:
        """向共享记忆写入（带权限检查）。"""
        return self.shared_memory.write(self.agent_id, namespace, key, value)

`run` 方法是智能体真正干活的地方。
它从共享记忆和私人草稿本收集上下文，然后把所有内容发给 LLM。
注意它只读取智能体有权访问的命名空间。

In [ ]:
def _run(self, task: str, context_namespaces: list[str] | None = None) -> str:
    """执行任务，用共享记忆作为 LLM 的上下文。

    步骤：
    1. 收集共享记忆中可读的条目。
    2. 加入私人草稿本笔记。
    3. 把合并后的上下文发给 LLM。
    4. 返回响应。"""
    # 构建共享记忆上下文
    shared_entries = self.shared_memory.read_all_accessible(
        self.agent_id, context_namespaces
    )
    shared_context = ""
    if shared_entries:
        shared_context = "## Shared Memory (visible to all agents)\n\n"
        for entry in shared_entries:
            shared_context += (
                f"**[{entry.namespace}/{entry.key}]** (by {entry.author}):\n"
                f"{entry.value}\n\n"
            )

    # 构建私人草稿本上下文
    scratchpad_context = ""
    if self.scratchpad:
        scratchpad_context = "## Your Private Scratchpad\n\n"
        for note in self.scratchpad:
            scratchpad_context += f"- {note}\n"
        scratchpad_context += "\n"

    user_message = f"{shared_context}{scratchpad_context}## Current Task\n\n{task}"

    response = llm.invoke([
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_message},
        ])

    return to_text(response.content)

def ___repr__(self) -> str:
    return f"Agent('{self.agent_id}', role='{self.role}')"

Agent.run = _run

Agent.__repr__ = ___repr__


### 黑板

`Blackboard` 提供所有共享记忆的只读聚合视图。智能体通过它查看协作工作的当前状态，不必逐个查询命名空间。

In [ ]:
class Blackboard:
    """把共享记忆聚合为统一的只读视图。"""

    def __init__(self, shared_memory: SharedMemoryPool):
        self.shared_memory = shared_memory

    def get_summary(self, agent_id: str) -> str:
        """为智能体能看到的所有内容生成文本摘要。"""
        entries = self.shared_memory.read_all_accessible(agent_id)
        if not entries:
            return "The blackboard is empty. No entries posted yet."

        sections: dict[str, list[str]] = {}
        for entry in entries:
            sections.setdefault(entry.namespace, []).append(
                f"  [{entry.key}] (v{entry.version}, by {entry.author}): "
                f"{entry.value[:120]}"
            )

        lines = ["Current Blackboard State:", ""]
        for ns in sorted(sections):
            lines.append(f"=== {ns}/ ===")
            lines.extend(sections[ns])
            lines.append("")

        return "\n".join(lines)

## 示例运行

我们将搭建一个三智能体团队，协作完成一个编程任务：

- **研究员（Researcher）**：收集需求。写入 `research/`，可读所有内容。
- **程序员（Coder）**：编写代码。写入 `code/`，可读所有内容。
- **审查员（Reviewer）**：审查代码。写入 `review/`，可读所有内容。

每个智能体只写入自己的命名空间，但可以读取所有命名空间。这样既允许完全可见，又防止意外覆盖。

### 配置权限并创建智能体

我们先设置访问控制规则，再创建智能体团队。

In [ ]:
# 创建访问控制层
acl = AccessControl()

# 研究员：写入 research/，可读所有内容
acl.grant("researcher", "research", Permission.READ_WRITE)
acl.grant("researcher", "code", Permission.READ)
acl.grant("researcher", "review", Permission.READ)

# 程序员：写入 code/，可读所有内容
acl.grant("coder", "research", Permission.READ)
acl.grant("coder", "code", Permission.READ_WRITE)
acl.grant("coder", "review", Permission.READ)

# 审查员：写入 review/，可读所有内容
acl.grant("reviewer", "research", Permission.READ)
acl.grant("reviewer", "code", Permission.READ)
acl.grant("reviewer", "review", Permission.READ_WRITE)

print("Permission matrix:")
print(acl.describe())

现在创建共享记忆池、黑板和三个智能体。
每个智能体都有一个针对其角色的系统提示词（system prompt），用来引导它的行为。
共享记忆池通过上面定义的访问控制层连接所有智能体。

In [ ]:
# 创建共享记忆池
shared_memory = SharedMemoryPool(acl, conflict_strategy="last_write_wins")

# 创建黑板视图
blackboard = Blackboard(shared_memory)

# 创建智能体
researcher = Agent(
    agent_id="researcher",
    role="Research Specialist",
    system_prompt=(
        "You are a research specialist. Analyze requirements and list "
        "the key technical details. Be specific and concise. "
        "Write no more than 8 bullet points."
    ),
    shared_memory=shared_memory,
)

coder = Agent(
    agent_id="coder",
    role="Software Engineer",
    system_prompt=(
        "You are a software engineer. Write clean, working Python code "
        "based on the research findings in shared memory. "
        "Keep code short and well-commented. Include example usage."
    ),
    shared_memory=shared_memory,
)

reviewer = Agent(
    agent_id="reviewer",
    role="Code Reviewer",
    system_prompt=(
        "You are a code reviewer. Review the code for correctness, "
        "clarity, and edge cases. Be constructive. Point out specific "
        "issues and suggest fixes. Keep feedback to 5 items or fewer."
    ),
    shared_memory=shared_memory,
)

print(f"\nAgents created: {researcher}, {coder}, {reviewer}")

### 运行协作任务

三个智能体按顺序处理同一个任务。每个智能体从共享记忆读取前面智能体的贡献，完成自己的工作，再把结果发布回去。

任务：编写一个在摄氏度、华氏度和开尔文之间转换温度的 Python 函数。

In [ ]:
task = "Build a Python function that converts temperatures between Celsius, Fahrenheit, and Kelvin."

print("=" * 60)
print(f"TASK: {task}")
print("=" * 60)

# 第 1 步：研究员收集需求
print("\n--- Step 1: Researcher gathers requirements ---\n")
research_output = researcher.run(
    f"Research the requirements for this task: {task}\n\n"
    "List the conversion formulas, edge cases, and design considerations."
)
print(f"Researcher:\n{research_output}\n")

# 研究员把发现发布到共享记忆
researcher.write_shared("research", "requirements", research_output)
researcher.think("Posted requirements to research/requirements")  # 私人笔记

# 第 2 步：程序员编写实现
print("\n--- Step 2: Coder writes implementation ---\n")
coder_output = coder.run(
    "Write a Python implementation based on the research requirements "
    "in shared memory. Include the function and example calls.",
    context_namespaces=["research"],
)
print(f"Coder:\n{coder_output}\n")

# 程序员把代码发布到共享记忆
coder.write_shared("code", "implementation", coder_output)

# 第 3 步：审查员检查工作
print("\n--- Step 3: Reviewer checks the code ---\n")
reviewer_output = reviewer.run(
    "Review the code in shared memory against the research requirements. "
    "List any bugs, missing edge cases, or improvements.",
    context_namespaces=["research", "code"],
)
print(f"Reviewer:\n{reviewer_output}")

# 审查员把反馈发布到共享记忆
reviewer.write_shared("review", "feedback", reviewer_output)

### 检查共享记忆状态

智能体完成后，我们可以查看完整黑板和操作历史。历史是一条审计线索，记录每个智能体在何时贡献了什么。

In [ ]:
# 查看完整黑板
print("=" * 60)
print("FINAL BLACKBOARD STATE")
print("=" * 60)
print(blackboard.get_summary("reviewer"))  # 审查员可以看到所有内容

# 查看审计日志
print("\n--- Operation History ---")
for op in shared_memory.get_history():
    print(f"  {op['action'].upper()} {op['key']} (v{op['version']}) by {op['agent']}")

### 私人记忆与共享记忆

研究员在上一步添加了一条私人草稿本笔记。我们来确认私人笔记保持私密，不会出现在共享记忆中。

In [ ]:
# 研究员的私人草稿本
print("Researcher's private scratchpad:")
for note in researcher.scratchpad:
    print(f"  - {note}")

# 程序员和审查员的草稿本是空的
print(f"\nCoder's scratchpad: {coder.scratchpad}")
print(f"Reviewer's scratchpad: {reviewer.scratchpad}")

# 共享记忆中只有显式写入的内容
print(f"\nEntries in shared memory: {len(shared_memory._store)}")
print("Private scratchpad notes do NOT appear in shared memory.")

### 权限强制执行

每个智能体只能写入自己的命名空间。向其他智能体的命名空间写入会抛出 `PermissionError`。访问控制就是这样防止未授权修改的。

In [ ]:
# 研究员不能写入 code/ 命名空间
try:
    researcher.write_shared("code", "sneaky_write", "This should fail")
    print("ERROR: Write should have been blocked!")
except PermissionError as e:
    print(f"Blocked (expected): {e}")

# 程序员不能写入 research/ 命名空间
try:
    coder.write_shared("research", "sneaky_write", "This should also fail")
    print("ERROR: Write should have been blocked!")
except PermissionError as e:
    print(f"Blocked (expected): {e}")

print("\nPermission enforcement works correctly.")

### 冲突解决

当两个智能体写入同一个键时，我们需要一个策略。来比较一下后写胜出和版本检查两种方案。

**后写胜出（last-write-wins）**是默认策略。最近的写入覆盖旧值。速度快，但可能丢数据。

**版本检查（version check）**（乐观锁）要求写入者知道当前版本。如果你上次读取之后有别人更新了条目，写入就会失败。它把冲突暴露出来，而不是悄悄丢弃数据。

In [ ]:
# --- 后写胜出演示 ---
print("=== Last-Write-Wins Strategy ===\n")

demo_acl = AccessControl()
demo_acl.grant("agent_a", "shared", Permission.READ_WRITE)
demo_acl.grant("agent_b", "shared", Permission.READ_WRITE)

lww_memory = SharedMemoryPool(demo_acl, conflict_strategy="last_write_wins")

entry_a = lww_memory.write("agent_a", "shared", "status", "Task is 50% complete")
print(f"Agent A writes: v{entry_a.version} -> '{entry_a.value}'")

entry_b = lww_memory.write("agent_b", "shared", "status", "Task is 75% complete")
print(f"Agent B writes: v{entry_b.version} -> '{entry_b.value}'")

result = lww_memory.read("agent_a", "shared", "status")
print(f"Final value: '{result[0].value}' (by {result[0].author}, v{result[0].version})")
print("Agent B's write overwrote Agent A's value.\n")

# --- 版本检查演示 ---
print("=== Version Check Strategy ===\n")

vc_memory = SharedMemoryPool(demo_acl, conflict_strategy="version_check")

# 智能体 A 写入初始值
entry = vc_memory.write("agent_a", "shared", "status", "Draft v1")
print(f"Agent A writes: v{entry.version} -> '{entry.value}'")

# 智能体 B 读取 v1，然后以 expected_version=1 写入
entry_b = vc_memory.write(
    "agent_b", "shared", "status", "Revised by B",
    expected_version=1,
)
print(f"Agent B writes (expected v1): v{entry_b.version} -> '{entry_b.value}'")

# 智能体 A 试图用过期版本写入（预期 v1，但实际已是 v2）
try:
    vc_memory.write(
        "agent_a", "shared", "status", "Revised by A",
        expected_version=1,  # 过期：实际版本现在是 2
    )
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Conflict detected (expected): {e}")

print("\nVersion check catches stale writes instead of silently overwriting.")

## 权衡

### 多智能体共享记忆的适用场景

- **角色清晰、需要协作的流程。**当智能体各有明确职责（研究、编码、审查）时，共享记忆提供了干净的交接点。每个智能体写入自己的命名空间，读取别人的。
- **需要迭代完善的任务。**黑板模式让智能体在彼此的工作上继续构建。审查员的反馈可以触发程序员的修改。每一轮都让结果更好。
- **审计与调试。**操作历史清晰记录每个智能体贡献了什么。你可以追溯最终结果是如何由各个贡献组装出来的。
- **并行执行智能体。**线程安全的共享记忆让多个智能体同时工作。锁防止并发写入破坏数据。

### 什么时候会失效

- **高频并发写入。**如果很多智能体快速写入相同的键，锁会变成瓶颈。后写胜出会丢数据，版本检查会频繁触发重试。
- **记忆存储变大。**随着共享记忆增长，智能体每轮读取更多上下文，token 用量和成本随之上升。你可能需要基于检索的访问（搜索相关条目），而不是读取全部内容。
- **紧耦合智能体。**如果智能体需要实时依赖彼此的输出（像一来一回的对话），共享记忆反而增加开销。对紧耦合流程，直接消息传递可能更快。
- **安全敏感数据。**这里的权限模型运行在应用代码里。一个 bug 就可能把私人草稿本内容泄漏到共享记忆。生产环境需要更强的隔离（例如独立进程或加密）。

## 延伸阅读

- Nii, H. P. (1986). ["Blackboard Systems."](https://doi.org/10.1609/aimag.v7i2.537) *AI Magazine*, 7(2), 38-53。多智能体协作黑板架构的奠基论文。

- [LangGraph 多智能体模式](https://langchain-ai.github.io/langgraph/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：LangGraph 用共享状态图管理多智能体状态的方法。

- [CrewAI 共享记忆](https://docs.crewai.com/concepts/memory?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：CrewAI 在协作智能体团队中对共享记忆和每智能体记忆的实现。

- [AutoGen 多智能体对话](https://microsoft.github.io/autogen/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：微软的多智能体对话框架，采用结构化消息协议。

---

*← 上一篇：[21：跨会话记忆](../21_cross_session_memory/) · 下一篇：[23：工具化记忆](../23_memory_with_tools/) →*

## 🧪 自己动手试试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：新的专职智能体
为系统添加一个 'debugger' `Agent`，给它自己的命名空间，并通过 `AccessControl` 授予对 researcher 和 coder 命名空间的只读访问。运行一个三智能体流程，验证 debugger 能读取但不能写入其他命名空间。

### 挑战 2：测量冲突率
让 3 个智能体使用 `SharedMemoryPool` 的版本检查冲突解决策略，快速连续写入同一个命名空间键。执行 20 次写入尝试，统计有多少次触发版本冲突。报告冲突率和所有写入完成后的最终值。

### 挑战 3：基于优先级的冲突解决
把后写胜出换成按重要性打分的解决策略。每次写入附带一个优先级分数；发生冲突时，`SharedMemoryPool` 保留优先级更高的值。用智能体以不同优先级写入相互矛盾的事实来测试。这个想法来自技术 14 记忆巩固中的重要性评分思路。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--22-multi-agent-shared-memory--multi-agent-shared-memory)